<a href="https://colab.research.google.com/github/mariacastrom00/AI_exercises_MJCM/blob/main/assignment4_MJCM_AH2179.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4 - Use the neural network model for bike-sharing demand prediction

By: Maria José Castro Moreno
Course: AH2179

## Data preparation


In [ ]:
import pandas as pd
import numpy as np

# Import for task 1
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Import for task 2
import matplotlib.pyplot as plt
from sklearn.decomposition import KernelPCA

# Import for task 3
from sklearn.datasets import make_classification
# pip install factor_analyzer
from factor_analyzer import FactorAnalyzer
import matplotlib.pyplot as plt
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo



#upload the dataset by downloading both datasets from canvas and upload it on colab
data_df = pd.read_csv("dataset_exercise_5_clustering_highway_traffic.csv",sep=";")
data_df

### Long wide transformation

In [ ]:
# Sort the DataFrame 'data_df' by columns "Date" and "Interval_5"
data_df.sort_values(["Date", "Interval_5"])
# Extract unique dates from the sorted DataFrame
days = np.unique(data_df[['Date']].values.ravel())
# Calculate the total number of unique days
ndays = len(days)
# Group the DataFrame 'data_df' by the "Date" column
day_subsets_df = data_df.groupby(["Date"])
# Define the total number of 5-minute intervals in a day
nintvals = 288

In [ ]:
# Create a matrix 'vectorized_day_dataset' filled with NaN values
vectorized_day_dataset = np.zeros((ndays, nintvals))
vectorized_day_dataset.fill(np.nan)
# Loop through each unique day
for i in range(0, ndays):
    # Get the DataFrame corresponding to the current day
    df_t = day_subsets_df.get_group(days[i],)
    # Loop through each row in the current day's DataFrame
    for j in range(len(df_t)):
        # Get the current day's DataFrame
        df_t = day_subsets_df.get_group(days[i],)
        # Extract the "Interval_5" and "flow" values and populate 'vectorized_day_dataset'
        vectorized_day_dataset[i, df_t.iloc[j]["Interval_5"]] = df_t.iloc[j]["flow"]
# Print the resulting 'vectorized_day_dataset' and the size of vector
print(vectorized_day_dataset)
print(vectorized_day_dataset.shape)

### Missing values

In [ ]:
# print the number of days with missing value
nans_per_day = np.sum(np.isnan(vectorized_day_dataset),1)
print('number of days with missing value:',np.size(np.where(nans_per_day > 0),1))

In [ ]:
# Drop the days with missing values
vectorized_day_dataset_no_nans = vectorized_day_dataset[np.where(nans_per_day == 0)[0],:]
# days_not_nans = days[np.where(nans_per_day == 0)[0]]
print(vectorized_day_dataset_no_nans.shape)

## Task 1: Perform PCA in dataset 1 for dimesionality reduction

### Overview of the dataset

In [ ]:
data = pd.DataFrame(vectorized_day_dataset_no_nans)
data.head()  # Display the first few rows

###Data preprocessing (Normalization).

StandardScaler transforms the data so that each feature has a mean of 0 and a standard deviation of 1.

In [ ]:
scaler = StandardScaler()
data_normalized = scaler.fit_transform(data)
data_normalized

###Perform PCA for dimensionality reduction

In [ ]:
pca = PCA(n_components=6)  # Adjust the number of components (dimensions) as needed
data_pca = pca.fit_transform(data_normalized)
print(data_pca)
print(data_pca.shape)

### Assess performance of PCA

In [ ]:
# Investigate Explained Variance
explained_variance = pca.explained_variance_ratio_
print(f'Explained Variance Ratio: {explained_variance}')
print(f"Sum of the explained variance: {sum(explained_variance)}")

In [ ]:
pca.components_

### Task 1.1: What is the potential issue of PCA in terms of interpretability of the results?

ANSWER HEREEEEEE

###Task 1.2: Hoe many dimensions do we need to preserve at least 85% of the variance?

ANSWER HEREEEE

###Stability analysis for PCA

In [ ]:
# Conduct Stability Analysis
num_runs = 10  # Number of times to run PCA with different random seeds

prop_data_used = 0.8 # Define the proportion of data to use in each iteration (e.g., 80%)

data_normalized_df = pd.DataFrame(data_normalized)

for i in range(num_runs):
    # Randomly select a subset of the data
    prop_data_used
    subset_indices = np.random.choice(data_normalized_df.shape[0], size=int(prop_data_used * data_normalized_df.shape[0]), replace=False)
    subset_data = data_normalized_df.loc[subset_indices]
    # Fit PCA on the subset
    pca = PCA(n_components=10)  # Change random_state for each run
    X_pca = pca.fit_transform(subset_data)

    explained_variance = pca.explained_variance_ratio_
    print(f'Run {i+1} - Explained Variance Ratio: {explained_variance}, Sum: {sum(explained_variance)}')

### Task 1.3: Interpret the stability analysis of the PCA


ANSWER HEREEEEE

## Perform PCA in dataset 1 for outlier detection

In [ ]:
# Load and preprocess the data
data = pd.DataFrame(vectorized_day_dataset_no_nans)
scaler = StandardScaler()
X_normalized = scaler.fit_transform(data)

# Apply PCA for reconstruction
pca = PCA(n_components=2)
X_dimensionality_reduced = pca.fit_transform(X_normalized)
X_reconstructed = pca.inverse_transform(X_dimensionality_reduced)

# Calculate reconstruction errors
reconstruction_error = np.mean(np.square(X_normalized - X_reconstructed), axis=1)

# Set a threshold for the reconstruction error
threshold = 1  # Adjust as needed

# Identify outliers
outliers = np.where(reconstruction_error > threshold)[0]

# Visual Inspection of PCA Score Plots
pca_scores = pca.fit_transform(X_normalized)

plt.figure(figsize=(8, 6))
plt.scatter(pca_scores[:, 0], pca_scores[:, 1], c='b', alpha=0.5, label='Normal')
plt.scatter(pca_scores[outliers, 0], pca_scores[outliers, 1], c='r', label='Outliers')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.title('PCA Score Plot')
plt.show()

# Print the indices of detected outliers
print(f'Number of detected outliers: {outliers.shape[0]}')
print(f'Detected outliers: {outliers}')

### Using kernel PCA

In [ ]:
# Load and preprocess the data
data = pd.DataFrame(vectorized_day_dataset_no_nans)
scaler = StandardScaler()
X_normalized = scaler.fit_transform(data)

# Step 1: Perform Kernel PCA
# Kernel functions : 'linear', 'rbf', 'poly', 'sigmoid'.
kpca = KernelPCA(kernel='linear', n_components=2, fit_inverse_transform=True)
X_kpca = kpca.fit_transform(X_normalized)

# Step 2: Detecting Outliers through Computing Reconstruction Errors
# Reconstruct the data from the reduced kernel PCA space
X_reconstructed = kpca.inverse_transform(X_kpca)

# Calculate the reconstruction error
reconstruction_error = np.mean(np.square(X_normalized - X_reconstructed), axis=1)

# Define a threshold for outliers
threshold = 1  # Adjust as needed
kpca_scores = kpca.fit_transform(X_normalized)
# Identify outliers
outliers = np.where(reconstruction_error > threshold)[0]

plt.figure(figsize=(8, 6))
plt.scatter(kpca_scores[:, 0], kpca_scores[:, 1], c='b', alpha=0.5, label='Normal')
plt.scatter(kpca_scores[outliers, 0], kpca_scores[outliers, 1], c='r', label='Outliers')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.title('KPCA Score Plot')
plt.show()
# Print the number of detected outliers
print(f'Number of detected outliers: {outliers.shape[0]}')
print(f'Detected outliers: {outliers}')

### Task 2.1: Why does the Kernel PCA detects the same outliers as the standard PCA when using a linear kernel

ASNWER HEREEEE

### Task 2.2: Do you have and idea how we could decide if we should use PCA, kernel PCA and what kernel of kernel PCA?

ANSWER HEREEEEEE

## Task 3: Identify the underlying latent factors that explain the observed correlations among several variables using FA

In [ ]:
# Step 1: Data Preprocessing
# Load and preprocess the data
data = pd.read_csv('Dataset 2.csv')

# Step 2: Exploratory Factor Analysis
# Initialize FactorAnalyzer object
fa = FactorAnalyzer()

# # Experiment with different rotation methods (e.g., Varimax, Promax, and Oblimin)
# # Varimax Rotation
# fa.set_params(rotation='varimax')
# fa.fit(data)

# # Promax Rotation
# fa.set_params(rotation='promax')
# fa.fit(data)

# # Oblimin Rotation
fa.set_params(rotation='oblimin')
fa.fit(data)

# Step 3: Determine the Number of Factors
# Kaiser's Criterion
eigenvalues, _ = fa.get_eigenvalues()
num_factors_kaiser = sum(eigenvalues > 1)  # Select factors with eigenvalues > 1

# Scree Plot Examination
plt.scatter(range(1, data.shape[1]+1), eigenvalues)
plt.plot(range(1, data.shape[1]+1), eigenvalues)
plt.title('Scree Plot')
plt.xlabel('Factor Number')
plt.ylabel('Eigenvalue')
plt.show()

### Task 3.1: Identify how many factors are in the dataset

In [ ]:
# Step 4: Compute Factor Loadings
fa.set_params(n_factors=num_factors_kaiser)  # Use the determined number of factors
fa.fit(data)
factor_loadings = fa.loadings_

# Step 5: Interpret the Factors
# (Based on factor loadings and variable names)
# Provide meaningful labels/names to the identified factors based on interpretation
factor_names = [f"Factor {id+1}" for id in range(0,num_factors_kaiser)]  # Adapt as needed

# Print factor loadings
factor_loadings = pd.DataFrame(factor_loadings, index=data.columns, columns=factor_names)
print("Factor Loadings:", factor_loadings)
# print(pd.DataFrame(factor_loadings, index=data.columns, columns=factor_names))

# You can further interpret and label the factors based on the loadings and variable names.

# Additional: Assessing Factorability
bartlett_test_statistic, p_value = calculate_bartlett_sphericity(data)
kmo_score = calculate_kmo(data)

print(f"\nBartlett's Test Statistic: {bartlett_test_statistic}")
print(f"P-value: {p_value}")
print(f"KMO Score: {kmo_score}")

### Task 3.2: By analysing the factor loadings, specify for each factor what variables it influences

ANSWER HEREEEEE

### Task 3.3: Compare the performance of PCA and kernel PCA for dimensionality reduction in dataset 1

Your task is to do the following:

Experiment with different kernel functions (e.g., Gaussian, polynomial, etc.) and assess their performance for dimensionality reduction.
Pick the most suitable kernel function to perform kernel PCA for dimensionality reduction of dataset 1.
Compare the performance of kernel PCA with PCA (baseline from practice task 1) for dimensionality reduction in dataset 1 and propose a suitable method to perform dimensionality reduction in dataset 1.
Provide a reflection on the dimensionality reduction results obtained using PCA and kernel PCA. Was PCA sufficient to capture the data’s structure? Which one would you choose to perform your dimensionality reduction task and why?